In [ ]:
from pilot.models        import build_logdeeponet
from pilot.data          import ODEIterableDataset, DirichletSampler, ConstrainedLHCSampler
from pilot.physics       import Robertson
from pilot.training      import Trainer, build_dataloaders
from pilot.losses        import ScaledMSE, RobertsonPhysicsLoss

import numpy as np
import matplotlib.pyplot as plt 

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
# Initialize Roberston Model
k1 = 4e-2
k2 = 3e7
k3 = 1e4
system = Robertson([k1, k2, k3])

# Initialize Sampler Object 
sampler = ConstrainedLHCSampler(low=0.8)

In [10]:
t_span  = (1e-4, 1e6)

# Initialize Training and validation datasets
train_dataset    = ODEIterableDataset(size = 2000,
                                      system_class = system,
                                      sampler      = sampler,
                                      t_span       = t_span,
                                      log_sampling = True,
                                      method       = "BDF",
                                      output_mask  = None)

val_dataset      = ODEIterableDataset(size = 100,
                                      system_class = system,
                                      sampler      = sampler,
                                      t_span       = t_span,
                                      log_sampling = True,
                                      method       = "BDF", 
                                      output_mask  = None)

In [11]:
# Set Up DeepONet configuration 

"""
DEEPONET_CONFIG = {
    
    "hidden_size" : 64,
    "depth"       : 4,
    "latent_size" : 60,
    "input_size_b": 3,
    "input_size_t": 1,
    "output_size" : 3,
    "activation"  : "gelu",
    "t_span"      : t_span

}
"""
DEEPONET_CONFIG = {
    
    "hidden_size" : 128,
    "depth"       : 4,
    "latent_size" : 120,
    "input_size_b": 3,
    "input_size_t": 1,
    "output_size" : 3,
    "activation"  : "gelu",
    "t_span"      : t_span

}


# Initialize DeepONet network 

deeponet = build_logdeeponet(DEEPONET_CONFIG)

In [12]:
# Set Up training Configuration as well as optimizer and loss functions

"""
TRAIN_CONFIG = {

    "num_epochs"     : 1000,
    "learning_rate"  : 0.000034728,
    "batch_size"     : 64,
    "num_workers"    : 12,
    "Save_model"     : True,
    "Save_directory" : "../../weights/best_robertson_4.pth"
}
"""


TRAIN_CONFIG = {

    "num_epochs"     : 51,
    "learning_rate"  : 0.0005167999414530766,
    "batch_size"     : 64,
    "num_workers"    : 12,
    "Save_model"     : True,
    "Save_directory" : "../../weights/best_robertson_5.pth"
}


optimizer = torch.optim.Adam

In [ ]:
y_scale = [1, 3.6e-5, 1]
loss_fn = RobertsonPhysicsLoss(k1, k2, k3, yscale, lambda_data=0.5, lambda_cons=1.2)

# Move to device along with your model
loss_fn = loss_fn.to(device)

In [ ]:
# Initialize Train Object and Train 

trainer   = Trainer(
    model         = deeponet,
    train_dataset = train_dataset, 
    val_dataset   = val_dataset, 
    optimizer     = optimizer,
    loss_fn       = loss_fn, 
    train_cfg     = TRAIN_CONFIG,
    model_cfg     = DEEPONET_CONFIG,
    project       = "PINN_rober",
    mode          = "online" 
)

trainer.run()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


train_loss,█▆▅▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss,-2.85849
val_loss,0.04137


Training: 100%|██████████| 51/51 [02:35<00:00,  3.05s/it]
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
